# `StructuredOutput`: field-level scoring of structured output

`Equals` scores structured output with whole-object `==`, so it is 0.0 or 1.0. `StructuredOutput`
scores the same output field by field, so the number says *how* wrong it is and *which* field to fix.

Offline and deterministic: no credentials and no model calls, so the extraction is stubbed here.
Scoring comes from [`stickler-eval`](https://pypi.org/project/stickler-eval/), installed via the
`stickler` extra:

```bash
pip install "strands-agents-evals[stickler]"
```


## Setup

In [1]:
import datetime

from pydantic import BaseModel, Field
from strands_evals import Case, Experiment, eval_task
from strands_evals.evaluators import Equals, StructuredOutput


## The model

A plain Pydantic model, the kind an agent already uses for `structured_output_model`. No stickler
annotations. Nullable fields are `Optional` because on a real invoice they are legitimately absent,
and "correctly returned nothing" has to score as a success.

In [2]:
class LineItem(BaseModel):
    sku: str | None = None
    description: str | None = None
    unit_price: float | None = None


class Invoice(BaseModel):
    invoice_id: str
    vendor_name: str
    invoice_date: datetime.date | None = None
    total_amount: float | None = None
    line_items: list[LineItem] = Field(default_factory=list)

## Cases

Six invoices, each broken a different way, so every confusion-matrix category shows up: a missed
field, an invented line item, several wrong values, and two that should score perfectly.

`expected_output` is the label. `metadata` carries the stubbed extraction that the task returns.

In [3]:
def inv(iid, vendor, date, total, items):
    return Invoice(invoice_id=iid, vendor_name=vendor, invoice_date=date,
                   total_amount=total, line_items=[LineItem(**i) for i in items])


CASES = [
    # (name, ground truth, prediction)
    ("perfect",
     inv("INV-001", "Acme Corporation", "2026-01-15", 150.00,
         [{"sku": "SKU-1", "description": "Widget", "unit_price": 150.00}]),
     inv("INV-001", "Acme Corporation", "2026-01-15", 150.00,
         [{"sku": "SKU-1", "description": "Widget", "unit_price": 150.00}])),
    ("case-only",                                   # vendor differs only in case
     inv("INV-002", "Beta Industries", "2026-02-01", 220.00,
         [{"sku": "SKU-2", "description": "Gadget", "unit_price": 220.00}]),
     inv("INV-002", "BETA INDUSTRIES", "2026-02-01", 220.00,
         [{"sku": "SKU-2", "description": "Gadget", "unit_price": 220.00}])),
    ("amount-off",                                  # total slightly wrong
     inv("INV-003", "Gamma Ltd", "2026-02-20", 310.00,
         [{"sku": "SKU-3", "description": "Sprocket", "unit_price": 310.00}]),
     inv("INV-003", "Gamma Ltd", "2026-02-20", 311.50,
         [{"sku": "SKU-3", "description": "Sprocket", "unit_price": 310.00}])),
    ("missing",                                     # date not extracted -> FN
     inv("INV-004", "Delta LLC", "2026-03-05", 90.00,
         [{"sku": "SKU-4", "description": "Cog", "unit_price": 90.00}]),
     inv("INV-004", "Delta LLC", None, 90.00,
         [{"sku": "SKU-4", "description": "Cog", "unit_price": 90.00}])),
    ("extra-line",                                  # hallucinated row -> FA
     inv("INV-005", "Epsilon SA", "2026-03-19", 400.00,
         [{"sku": "SKU-5", "description": "Flange", "unit_price": 400.00}]),
     inv("INV-005", "Epsilon SA", "2026-03-19", 400.00,
         [{"sku": "SKU-5", "description": "Flange", "unit_price": 400.00},
          {"sku": "SKU-9", "description": "Phantom", "unit_price": 12.00}])),
    ("wrong",                                       # wrong on nearly everything
     inv("INV-006", "Zeta Holdings", "2026-04-02", 75.00,
         [{"sku": "SKU-6", "description": "Bracket", "unit_price": 75.00}]),
     inv("INV-999", "Omega Group", "2025-11-11", 12.00,
         [{"sku": "SKU-X", "description": "Unrelated", "unit_price": 12.00}])),
]

cases = [
    Case[str, Invoice](
        name=name,
        input=f"(OCR text for {name})",
        expected_output=ground_truth,
        metadata={"category": "invoice", "stub": prediction},
    )
    for name, ground_truth, prediction in CASES
]

print(f"{len(cases)} cases")

6 cases


## The task

`@eval_task()` wraps the function the harness calls per case. A real task would build an `Agent` here
and return its structured output; this one replays a stub so the notebook stays offline and both
evaluators see identical predictions.

Returning a `dict` matters: `EvalTaskHandler` passes a dict through untouched but calls `str()` on
anything else, which would flatten a Pydantic model into text.

In [4]:
@eval_task()
def extract(case):
    return {"output": case.metadata["stub"]}

## Equals versus stickler

Same cases, same harness, same task. Only the evaluator differs.

In [5]:
evaluator = StructuredOutput(Invoice)

stickler_report = await Experiment[str, Invoice](
    cases=cases, evaluators=[evaluator]
).run_evaluations_async(extract)

equals_report = await Experiment[str, Invoice](
    cases=cases, evaluators=[Equals()]
).run_evaluations_async(extract)

print(f"{'case':12} {'stickler':>9} {'equals':>7}")
print("-" * 30)
for c, s, e in zip(stickler_report.cases, stickler_report.scores, equals_report.scores):
    print(f"{c['name']:12} {s:>9.3f} {e:>7.1f}")

print(f"\noverall      {stickler_report.overall_score:>9.3f} {equals_report.overall_score:>7.3f}")
print(f"distinct     {len(set(round(v, 4) for v in stickler_report.scores)):>9} "
      f"{len(set(round(v, 4) for v in equals_report.scores)):>7}")

case          stickler  equals
------------------------------
perfect          1.000     1.0
case-only        1.000     0.0
amount-off       0.800     0.0
missing          0.800     0.0
extra-line       0.900     0.0
wrong            0.025     0.0

overall          0.754   0.167
distinct             4       2


`Equals` collapses five of six to 0.0, including the one that differed only in capitalisation, so it
cannot rank extractors or spot a regression. Stickler separates "one field slightly off" from "wrong
on everything".

The report renders itself too. Use `display()` in a notebook; `run_display()` is the interactive
variant and blocks waiting for keyboard input.

In [6]:
stickler_report.display(include_input=False)

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.75           Pass Rate: 0.8333333333333334                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                          Test Case Results                           
┏━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ index ┃ name       ┃ evaluator        ┃ score ┃ test_pass ┃ reason ┃
┡━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ ▶ 0   │ perfect    │ StructuredOutput │ 1.00  │ ✅        │ ...    │
├───────┼────────────┼──────────────────┼───────┼───────────┼────────┤
│ ▶ 1   │ case-only  │ StructuredOutput │ 1.00  │ ✅        │ ...    │
├───────┼────────────┼──────────────────┼───────┼───────────┼────────┤
│ ▶ 2   │ amount-off │ StructuredOutput │ 0.80  │ ✅        │ ...    │
├───────┼────────────┼──────────────────┼───────┼───────────┼────────┤
│ ▶ 3   │ missing    │ StructuredOutput │ 0.80  │ ✅        │ ...    │
├───────┼────────────┼──────────────────┼───────┼───────────┼────────┤
│ ▶ 4   │ extra-line │ StructuredOutput │ 0.90  │ ✅        │ ...    │
├───────┼────────────┼──────────────────┼───────┼───────────┼────────┤
│ ▶ 5   │ wrong      │ StructuredOutput │ 0.03  │ ❌        │ ...    │
└───────┴────────────┴──────────────────┴───────┴───────────┴────────┘

## Per-case detail

`evaluate()` returns one `EvaluationOutput` per case carrying the weighted score. Field-level detail
comes from `per_case()`, read from the comparison the evaluator already performed. `EvaluationOutput`
has four scalar fields, so it could not carry this anyway.

In [7]:
for entry in evaluator.per_case():
    weak = {f: round(s, 2) for f, s in entry["field_scores"].items() if s < 1.0}
    print(f"{entry['case']:12} {entry['overall_score']:>5.2f}  "
          f"pass={str(entry['matched']):5}  weak={weak or '-'}")

perfect       1.00  pass=True   weak=-
case-only     1.00  pass=True   weak=-
amount-off    0.80  pass=True   weak={'total_amount': 0.0}
missing       0.80  pass=True   weak={'invoice_date': 0.0}
extra-line    0.90  pass=True   weak={'line_items': 0.5}
wrong         0.03  pass=False  weak={'invoice_id': 0.0, 'vendor_name': 0.0, 'invoice_date': 0.0, 'total_amount': 0.0, 'line_items': 0.12}


## `test_pass` is a document verdict, not a per-field one

Look at `amount-off` above: `pass=True` with `total_amount: 0.0`. That is not a bug, but it will
surprise you if you gate a suite on `test_pass` alone.

`test_pass` is the *weighted document score* against `match_threshold` (0.7 by default). Fields are
weighted uniformly, so on a five-field invoice one entirely wrong field costs 0.2 and the document
still clears 0.7. A wrong invoice total is exactly the kind of error you do not want to discover this
way.

Three ways to make it visible, in increasing strictness:


In [8]:
# 1. Raise match_threshold so less slack is tolerated per document.
strict = StructuredOutput(Invoice, match_threshold=0.95)
strict_report = await Experiment[str, Invoice](
    cases=cases, evaluators=[strict]
).run_evaluations_async(extract)

print(f"{'case':12} {'score':>6} {'thr=0.7':>8} {'thr=0.95':>9}")
print('-' * 39)
for entry, loose, tight in zip(strict.per_case(), stickler_report.scores, strict_report.scores):
    base = next(e for e in evaluator.per_case() if e['case'] == entry['case'])
    print(f"{entry['case']:12} {entry['overall_score']:>6.2f} "
          f"{str(base['matched']):>8} {str(entry['matched']):>9}")

# 2. Gate on the fields you actually care about, from per_case().
MUST_BE_EXACT = ('invoice_id', 'total_amount')
print('\nper-field gate on', MUST_BE_EXACT)
for entry in evaluator.per_case():
    bad = {f: s for f, s in entry['field_scores'].items() if f in MUST_BE_EXACT and s < 1.0}
    verdict = 'FAIL' if bad else 'ok'
    print(f"  {entry['case']:12} {verdict:5} {bad if bad else ''}")

# 3. Read the rollup: fd counts every wrong field, whatever the document verdict said.
print('\nrollup fd per field (document verdicts cannot hide these)')
fd_rollup = evaluator.metrics()['Invoice']
for name, row in sorted(fd_rollup.field_metrics.items()):
    if row['fd']:
        print(f"  {name:24} fd={row['fd']}")


case          score  thr=0.7  thr=0.95
---------------------------------------
perfect        1.00     True      True
case-only      1.00     True      True
amount-off     0.80     True     False
missing        0.80     True     False
extra-line     0.90     True     False
wrong          0.03    False     False

per-field gate on ('invoice_id', 'total_amount')
  perfect      ok    
  case-only    ok    
  amount-off   FAIL  {'total_amount': 0.0}
  missing      ok    
  extra-line   ok    
  wrong        FAIL  {'invoice_id': 0.0, 'total_amount': 0.0}

rollup fd per field (document verdicts cannot hide these)
  invoice_date             fd=1
  invoice_id               fd=1
  line_items               fd=1
  total_amount             fd=2
  vendor_name              fd=1


## Dataset rollup

`metrics()` returns stickler's five-category confusion matrix per field path, including nested paths.
It runs no extra comparisons: each case was compared once above and the raw result was kept.

**FN** is a field the extractor missed, **FA** one it invented, **FD** one it got wrong. A single
score cannot separate those, and they need different fixes.

In [9]:
rollup = evaluator.metrics()["Invoice"]

print(f"documents: {rollup.document_count}\n")
print(f"{'field':26} {'tp':>3} {'fn':>3} {'fa':>3} {'fd':>3}  {'prec':>5} {'rec':>5} {'f1':>5}")
print("-" * 62)
for path, m in sorted(rollup.field_metrics.items(),
                      key=lambda kv: (kv[1].get("cm_f1", 1.0), kv[0])):
    print(f"{path:26} {m.get('tp', 0):>3} {m.get('fn', 0):>3} {m.get('fa', 0):>3} {m.get('fd', 0):>3}"
          f"  {m.get('cm_precision', 0):>5.2f} {m.get('cm_recall', 0):>5.2f} {m.get('cm_f1', 0):>5.2f}")

documents: 6

field                       tp  fn  fa  fd   prec   rec    f1
--------------------------------------------------------------
total_amount                 4   0   0   2   0.67  1.00  0.80
invoice_date                 4   1   0   1   0.80  0.80  0.80
line_items                   5   0   1   1   0.71  1.00  0.83
invoice_id                   5   0   0   1   0.83  1.00  0.91
vendor_name                  5   0   0   1   0.83  1.00  0.91
line_items.description       5   0   0   0   1.00  1.00  1.00
line_items.sku               5   0   0   0   1.00  1.00  1.00
line_items.unit_price        5   0   0   0   1.00  1.00  1.00


Read the nested rows carefully. A `line_items.*` row only counts documents whose line-item pair scored
at or above `match_threshold`. Below that, threshold gating treats the pair as atomic and emits no
field breakdown, so those documents appear as `fd` on `line_items` and are absent from the child rows.
Child rows therefore have a smaller denominator than the document count.

Nested leaves carry counts and precision/recall/F1 but no mean score
([#249](https://github.com/awslabs/stickler/issues/249)). Row sets are data-dependent, so use `.get()`
rather than indexing.

## Why each field scored that way

Nothing was configured, so every comparator and threshold was inferred from the model. `explain()`
shows what was chosen and on what basis, which is what makes a score defensible.

In [10]:
print(f"{'field':26} {'comparator':24} {'thr':>5}  basis")
print("-" * 68)
for path, cfg in evaluator.explain().items():
    print(f"{path:26} {cfg['comparator']:24} {cfg['threshold']:>5}  {cfg['source']}")

field                      comparator                 thr  basis
--------------------------------------------------------------------
invoice_id                 ExactComparator            1.0  name-token
vendor_name                LevenshteinComparator     0.85  name-token
invoice_date               DateComparator            0.95  name-token
total_amount               NumericComparator         0.95  name-token
line_items                 Hungarian (per-element StructuredModel)   0.7  type
line_items.sku             ExactComparator            1.0  name-token
line_items.description     FuzzyComparator            0.6  name-token
line_items.unit_price      NumericComparator         0.95  name-token


## Mixed output types

`model_cls` is optional. Omit it and the model class is inferred per case, and `metrics()` partitions
its rollup by class. This matters because feeding two schemas into one rollup would union their field
paths, making a field present in half the documents look missed in the rest.

In [11]:
class Receipt(BaseModel):
    merchant: str
    tax: float


mixed = [
    Case[str, Invoice](name="inv-1", input="", expected_output=CASES[0][1],
                       metadata={"stub": CASES[0][2]}),
    Case[str, Receipt](name="rec-1", input="", expected_output=Receipt(merchant="Corner Store", tax=4.50),
                       metadata={"stub": Receipt(merchant="Corner Store", tax=9.99)}),
]

mixed_eval = StructuredOutput()          # no model_cls
await Experiment(cases=mixed, evaluators=[mixed_eval]).run_evaluations_async(extract)

for model_name, pe in mixed_eval.metrics().items():
    print(f"{model_name:10} docs={pe.document_count}  fields={sorted(pe.field_metrics)}")

Invoice    docs=1  fields=['invoice_date', 'invoice_id', 'line_items', 'line_items.description', 'line_items.sku', 'line_items.unit_price', 'total_amount', 'vendor_name']
Receipt    docs=1  fields=['merchant', 'tax']


Pass `model_cls` instead and the evaluator is strict: anything that will not validate as that class
raises, which is what a single-schema suite wants.

## Notes for reuse

The evaluator accumulates across cases, so call `reset()` before reusing an instance for a second run.

It is safe at any concurrency. During a run it only appends to a list, which is atomic under the GIL,
so it holds no locks and there is no need to pin `max_workers`. All aggregation happens afterwards.

In [12]:
print(f"before reset: {rollup.document_count} documents")
evaluator.reset()
print(f"after reset:  {evaluator.metrics()}  {evaluator.per_case()}")

before reset: 6 documents
after reset:  {}  []
